In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Profisee MDM Connector Simulator — Energy Transfer LP
# MAGIC
# MAGIC This notebook simulates the **Profisee MDM native connector for Databricks**,
# MAGIC generating realistic golden-record master data for a midstream energy company
# MAGIC modeled after **Energy Transfer LP**.
# MAGIC
# MAGIC ## What the Profisee Connector Does (Production)
# MAGIC
# MAGIC | Capability | Details |
# MAGIC |---|---|
# MAGIC | **Connection** | Host URL + SQL Endpoint + Access Token (no-code config) |
# MAGIC | **Export Modes** | On-Demand · Scheduled · Continuous |
# MAGIC | **Target** | Delta tables in Unity Catalog |
# MAGIC | **Metadata** | `_profisee_*` columns for lineage, DQ scores, match status |
# MAGIC | **Direction** | Bi-directional (read **and** write back to Profisee) |
# MAGIC
# MAGIC ## Simulated Master Data Domains
# MAGIC
# MAGIC | Table | Domain | Records |
# MAGIC |---|---|---|
# MAGIC | `dim_pipeline_assets` | Pipeline systems & segments | 200 |
# MAGIC | `dim_facilities` | Plants, terminals, storage | 120 |
# MAGIC | `dim_customers` | Utilities, industrials, marketers | 300 |
# MAGIC | `dim_vendors` | Suppliers & service providers | 180 |
# MAGIC | `dim_products` | Commodities & product specs | 50 |
# MAGIC | `dim_locations` | Geographic sites, GPS | 250 |
# MAGIC | `dim_employees` | Operators, stewards, contacts | 150 |
# MAGIC | `dim_regulatory` | Permits & compliance | 100 |
# MAGIC | `dim_contracts` | Transport & storage agreements | 200 |
# MAGIC | `fact_throughput_daily` | Daily volume measurements | 4,500 |

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 1 — Upload & Import the Simulator

# COMMAND ----------

# Upload profisee_energy_transfer_simulator.py to the workspace, then:
# %run ./profisee_energy_transfer_simulator

# Or paste the full script into a preceding cell.
# For this notebook, we import directly:
import sys, os

# If you placed the .py file in your workspace:
# sys.path.insert(0, "/Workspace/Users/<your-email>/")
from profisee_energy_transfer_simulator import ProfiseeMDMSimulator

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 2 — Generate All Master Data

# COMMAND ----------

sim = ProfiseeMDMSimulator(seed=42)
tables = sim.generate_all()
display(tables)



In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 3 — Write to Delta (Unity Catalog)

# COMMAND ----------

CATALOG = "mdm"
SCHEMA  = "energy_transfer"

sim.write_delta_tables(catalog=CATALOG, schema=SCHEMA)


In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 4 — Quick Exploration

# COMMAND ----------

display(spark.table(f"{CATALOG}.{SCHEMA}.dim_pipeline_assets").limit(20))

# COMMAND ----------

display(spark.table(f"{CATALOG}.{SCHEMA}.dim_customers").limit(20))

# COMMAND ----------

display(spark.table(f"{CATALOG}.{SCHEMA}.dim_facilities").limit(20))

# COMMAND ----------



In [0]:
# MAGIC %md
# MAGIC ## Cell 5 — Profisee Metadata Quality Analysis
# MAGIC
# MAGIC The `_profisee_*` columns let us audit the MDM process itself.

# COMMAND ----------

from pyspark.sql import functions as F

for table_name in ["dim_pipeline_assets", "dim_customers", "dim_facilities", "dim_vendors", "dim_contracts"]:
    df = spark.table(f"{CATALOG}.{SCHEMA}.{table_name}")
    print(f"\n{'='*60}")
    print(f"  {table_name}")
    print(f"{'='*60}")

    # Record status distribution
    print("\nRecord Status:")
    df.groupBy("_profisee_record_status").count().orderBy("count", ascending=False).show()

    # Match status (golden record vs matched vs unique)
    print("Match Status:")
    df.groupBy("_profisee_match_status").count().orderBy("count", ascending=False).show()

    # DQ score statistics
    print("Data Quality Score:")
    df.select(
        F.round(F.avg("_profisee_data_quality_score"), 2).alias("avg_dq"),
        F.round(F.min("_profisee_data_quality_score"), 2).alias("min_dq"),
        F.round(F.max("_profisee_data_quality_score"), 2).alias("max_dq"),
        F.round(F.stddev("_profisee_data_quality_score"), 2).alias("stddev_dq"),
    ).show()

    # Source system distribution
    print("Source Systems:")
    df.groupBy("_profisee_source_system").count().orderBy("count", ascending=False).show(5)



In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 6 — Lakeflow Declarative Pipeline (Silver + Gold Layers)
# MAGIC
# MAGIC Copy the SQL below into a **Lakeflow Declarative Pipeline** notebook to
# MAGIC build silver and gold layers on top of the simulated Profisee bronze data.
# MAGIC
# MAGIC This is exactly how you'd process real Profisee connector output in production.

# COMMAND ----------

# MAGIC %sql
# MAGIC -- ═══════════════════════════════════════════════════════════════════
# MAGIC -- SILVER LAYER: Cleansed, active golden records only
# MAGIC -- ═══════════════════════════════════════════════════════════════════
# MAGIC
# MAGIC CREATE OR REFRESH STREAMING TABLE silver_pipeline_assets
# MAGIC COMMENT 'Active, high-quality pipeline segments from Profisee MDM'
# MAGIC TBLPROPERTIES ('quality' = 'silver')
# MAGIC AS SELECT
# MAGIC     _profisee_code AS pipeline_code,
# MAGIC     pipeline_system,
# MAGIC     segment_id,
# MAGIC     business_segment,
# MAGIC     diameter_inches,
# MAGIC     length_miles,
# MAGIC     max_allowable_operating_pressure_psig AS maop_psig,
# MAGIC     pipe_material,
# MAGIC     commodity_transported,
# MAGIC     production_basin,
# MAGIC     state,
# MAGIC     installation_year,
# MAGIC     last_inline_inspection_date,
# MAGIC     capacity_mmbtu_per_day,
# MAGIC     capacity_barrels_per_day,
# MAGIC     is_interstate,
# MAGIC     ferc_regulated,
# MAGIC     _profisee_data_quality_score AS dq_score,
# MAGIC     _profisee_source_system AS source_system,
# MAGIC     _profisee_updated_dtm AS last_updated
# MAGIC FROM STREAM(mdm.energy_transfer.dim_pipeline_assets)
# MAGIC WHERE _profisee_record_status = 'Active'
# MAGIC   AND _profisee_data_quality_score >= 70;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH STREAMING TABLE silver_customers
# MAGIC COMMENT 'Active customer golden records from Profisee MDM'
# MAGIC TBLPROPERTIES ('quality' = 'silver')
# MAGIC AS SELECT
# MAGIC     _profisee_code AS customer_code,
# MAGIC     customer_name,
# MAGIC     customer_type,
# MAGIC     duns_number,
# MAGIC     city,
# MAGIC     state,
# MAGIC     credit_rating,
# MAGIC     contract_type,
# MAGIC     primary_commodity,
# MAGIC     annual_volume_contracted_mmbtu,
# MAGIC     payment_terms_days,
# MAGIC     is_key_account,
# MAGIC     _profisee_data_quality_score AS dq_score,
# MAGIC     _profisee_updated_dtm AS last_updated
# MAGIC FROM STREAM(mdm.energy_transfer.dim_customers)
# MAGIC WHERE _profisee_record_status = 'Active'
# MAGIC   AND _profisee_data_quality_score >= 70;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH STREAMING TABLE silver_facilities
# MAGIC COMMENT 'Operational facilities from Profisee MDM'
# MAGIC TBLPROPERTIES ('quality' = 'silver')
# MAGIC AS SELECT
# MAGIC     _profisee_code AS facility_code,
# MAGIC     facility_name,
# MAGIC     facility_type,
# MAGIC     business_segment,
# MAGIC     state,
# MAGIC     county,
# MAGIC     latitude,
# MAGIC     longitude,
# MAGIC     nameplate_capacity,
# MAGIC     capacity_uom,
# MAGIC     year_commissioned,
# MAGIC     connected_pipeline_system,
# MAGIC     _profisee_data_quality_score AS dq_score,
# MAGIC     _profisee_updated_dtm AS last_updated
# MAGIC FROM STREAM(mdm.energy_transfer.dim_facilities)
# MAGIC WHERE _profisee_record_status = 'Active'
# MAGIC   AND is_operational = true;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH STREAMING TABLE silver_contracts
# MAGIC COMMENT 'Active contracts from Profisee MDM'
# MAGIC TBLPROPERTIES ('quality' = 'silver')
# MAGIC AS SELECT
# MAGIC     _profisee_code AS contract_code,
# MAGIC     contract_number,
# MAGIC     contract_type,
# MAGIC     customer_code,
# MAGIC     pipeline_system,
# MAGIC     commodity,
# MAGIC     effective_date,
# MAGIC     expiration_date,
# MAGIC     term_years,
# MAGIC     max_daily_quantity_mmbtu,
# MAGIC     max_daily_quantity_bpd,
# MAGIC     rate_per_unit_usd,
# MAGIC     has_escalation_clause,
# MAGIC     status AS contract_status,
# MAGIC     _profisee_data_quality_score AS dq_score
# MAGIC FROM STREAM(mdm.energy_transfer.dim_contracts)
# MAGIC WHERE _profisee_record_status = 'Active'
# MAGIC   AND status IN ('Active', 'Pending Execution');

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH STREAMING TABLE silver_throughput
# MAGIC COMMENT 'Validated daily throughput measurements'
# MAGIC TBLPROPERTIES ('quality' = 'silver')
# MAGIC AS SELECT
# MAGIC     measurement_date,
# MAGIC     pipeline_asset_code,
# MAGIC     scheduled_volume_mmbtu,
# MAGIC     actual_volume_mmbtu,
# MAGIC     utilization_pct,
# MAGIC     flow_direction,
# MAGIC     measurement_quality_flag,
# MAGIC     _profisee_updated_dtm AS last_updated
# MAGIC FROM STREAM(mdm.energy_transfer.fact_throughput_daily)
# MAGIC WHERE measurement_quality_flag IN ('Validated', 'Estimated');

# COMMAND ----------

# MAGIC %sql
# MAGIC -- ═══════════════════════════════════════════════════════════════════
# MAGIC -- GOLD LAYER: Business-ready aggregations
# MAGIC -- ═══════════════════════════════════════════════════════════════════
# MAGIC
# MAGIC CREATE OR REFRESH MATERIALIZED VIEW gold_pipeline_summary AS
# MAGIC SELECT
# MAGIC     p.business_segment,
# MAGIC     p.commodity_transported,
# MAGIC     p.state,
# MAGIC     COUNT(*) AS segment_count,
# MAGIC     ROUND(SUM(p.length_miles), 1) AS total_miles,
# MAGIC     ROUND(AVG(p.dq_score), 2) AS avg_dq_score,
# MAGIC     SUM(CASE WHEN p.ferc_regulated THEN 1 ELSE 0 END) AS ferc_regulated_count
# MAGIC FROM silver_pipeline_assets p
# MAGIC GROUP BY p.business_segment, p.commodity_transported, p.state;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH MATERIALIZED VIEW gold_customer_portfolio AS
# MAGIC SELECT
# MAGIC     c.customer_type,
# MAGIC     c.state,
# MAGIC     COUNT(*) AS customer_count,
# MAGIC     SUM(c.annual_volume_contracted_mmbtu) AS total_contracted_volume,
# MAGIC     ROUND(AVG(c.dq_score), 2) AS avg_dq_score,
# MAGIC     SUM(CASE WHEN c.is_key_account THEN 1 ELSE 0 END) AS key_accounts
# MAGIC FROM silver_customers c
# MAGIC GROUP BY c.customer_type, c.state;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH MATERIALIZED VIEW gold_daily_utilization AS
# MAGIC SELECT
# MAGIC     t.measurement_date,
# MAGIC     p.pipeline_system,
# MAGIC     p.business_segment,
# MAGIC     p.commodity_transported,
# MAGIC     COUNT(*) AS segments_measured,
# MAGIC     ROUND(AVG(t.utilization_pct), 2) AS avg_utilization_pct,
# MAGIC     ROUND(SUM(t.actual_volume_mmbtu), 0) AS total_actual_mmbtu,
# MAGIC     ROUND(SUM(t.scheduled_volume_mmbtu), 0) AS total_scheduled_mmbtu
# MAGIC FROM silver_throughput t
# MAGIC JOIN silver_pipeline_assets p
# MAGIC   ON t.pipeline_asset_code = p.pipeline_code
# MAGIC GROUP BY t.measurement_date, p.pipeline_system, p.business_segment, p.commodity_transported;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE OR REFRESH MATERIALIZED VIEW gold_contract_exposure AS
# MAGIC SELECT
# MAGIC     cn.contract_type,
# MAGIC     cn.pipeline_system,
# MAGIC     cn.commodity,
# MAGIC     COUNT(*) AS contract_count,
# MAGIC     ROUND(AVG(cn.rate_per_unit_usd), 4) AS avg_rate_usd,
# MAGIC     SUM(cn.max_daily_quantity_mmbtu) AS total_firm_mdq_mmbtu,
# MAGIC     SUM(CASE WHEN cn.has_escalation_clause THEN 1 ELSE 0 END) AS with_escalation
# MAGIC FROM silver_contracts cn
# MAGIC WHERE cn.contract_status = 'Active'
# MAGIC GROUP BY cn.contract_type, cn.pipeline_system, cn.commodity;

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cell 7 — Data Quality Expectations (Optional)
# MAGIC
# MAGIC Add DQ constraints to the silver layer:

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Add to the silver_pipeline_assets definition:
# MAGIC -- CONSTRAINT valid_diameter   EXPECT (diameter_inches > 0)
# MAGIC -- CONSTRAINT valid_length     EXPECT (length_miles > 0)
# MAGIC -- CONSTRAINT valid_state      EXPECT (state IS NOT NULL)
# MAGIC -- CONSTRAINT high_dq_score    EXPECT (dq_score >= 70) ON VIOLATION DROP ROW

# COMMAND ----------

# MAGIC %md
# MAGIC ## Architecture Overview
# MAGIC
# MAGIC ```
# MAGIC ┌──────────────────┐     ┌─────────────────────────────────────────────────────┐
# MAGIC │  Source Systems   │     │              Databricks Lakehouse                   │
# MAGIC │                  │     │                                                     │
# MAGIC │  SAP ERP         │     │  ┌───────────┐  ┌────────────┐  ┌───────────────┐  │
# MAGIC │  Salesforce      │────▶│  │  BRONZE   │─▶│  SILVER    │─▶│    GOLD       │  │
# MAGIC │  Maximo EAM      │     │  │           │  │            │  │               │  │
# MAGIC │  SCADA           │     │  │ Profisee  │  │ Lakeflow   │  │ Materialized  │  │
# MAGIC │  GIS             │     │  │ Connector │  │ Declarative│  │ Views for     │  │
# MAGIC │  Legacy          │     │  │ Landing   │  │ Pipelines  │  │ BI / ML / AI  │  │
# MAGIC └────────┬─────────┘     │  │           │  │            │  │               │  │
# MAGIC          │               │  │ dim_*     │  │ silver_*   │  │ gold_*        │  │
# MAGIC          ▼               │  │ fact_*    │  │            │  │               │  │
# MAGIC ┌──────────────────┐     │  └───────────┘  └────────────┘  └───────────────┘  │
# MAGIC │   Profisee MDM   │     │                                                     │
# MAGIC │                  │     │  _profisee_* metadata columns flow through all      │
# MAGIC │  Match & Merge   │────▶│  layers for full lineage & auditability             │
# MAGIC │  DQ Validation   │     │                                                     │
# MAGIC │  Golden Records  │     └─────────────────────────────────────────────────────┘
# MAGIC │  Stewardship     │
# MAGIC └──────────────────┘
# MAGIC ```